In [1]:
pip install transformers datasets tensorflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

from datasets import load_dataset

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from transformers import BertTokenizer

/Users/jiheneguesmi/Desktop/machine_learning/sentiment/test2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
imdb = load_dataset("imdb")

train_df = pd.DataFrame(imdb["train"])
test_df = pd.DataFrame(imdb["test"])

train_df.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [4]:
# split features
X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]

In [5]:
# text cleaning
X_train = X_train.str.lower()
X_test = X_test.str.lower()

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer

max_words = 20000

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [9]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 200

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

print(X_train_pad.shape)

(25000, 200)


In [20]:
# bert from transformers import BertTokenizer

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [21]:
def encode_texts(texts, tokenizer, max_len=200):
    input_ids = []
    attention_masks = []

    for text in texts:
        encoded = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=max_len
        )

        input_ids.append(encoded["input_ids"])
        attention_masks.append(encoded["attention_mask"])

    return np.array(input_ids), np.array(attention_masks)

In [22]:
X_train_small = X_train[:5000]
X_test_small = X_test[:1000]

y_train_small = y_train[:5000]
y_test_small = y_test[:1000]

In [23]:
X_train_input_ids, X_train_attention = encode_texts(
    X_train_small,
    bert_tokenizer
)

X_test_input_ids, X_test_attention = encode_texts(
    X_test_small,
    bert_tokenizer
)

print("BERT input shape:", X_train_input_ids.shape)

BERT input shape: (5000, 200)


In [24]:
print("LSTM input:", X_train_pad.shape)
print("BERT input:", X_train_input_ids.shape)

LSTM input: (25000, 200)
BERT input: (5000, 200)


In [ ]:
## Preprocessing Summary

### LSTM Pipeline:
- Lowercasing
- Tokenization using Keras Tokenizer
- Integer sequences
- Padding to fixed length (200)

### BERT Pipeline:
- HuggingFace BertTokenizer
- Subword tokenization
- Attention masks automatically generated
- Fixed-length encoding

### Key Difference:
- LSTM uses word-level embeddings
- BERT uses contextual subword embeddings